In [19]:
# Section: Import Libraries

import os
import json
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style='whitegrid', context='talk')

OUT_DIR = Path('optimization/results/compare_runs/plots')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Ready. Plots will be written to', OUT_DIR)


Ready. Plots will be written to optimization/results/compare_runs/plots


# Title

Compare Preproc vs Greedy: analysis and plots

This notebook loads the per-seed JSON summaries produced by
`optimization/experiments/compare_preproc_vs_greedy.py` and the HDF5 outputs
sourced by those runs, and produces:
 - cumulative-best vs time (median + IQR)
 - ECDF of time-to-target (time to reach a given distance)
 - boxplots of final best LER
 - bar chart of median runtime / preprocessing overhead

Save plots to `optimization/results/compare_runs/plots/`.

In [20]:
# Section: Define Inputs B and C

# Directory containing compare-run JSONs
COMPARE_DIR = Path('optimization/results/compare_runs')
JSON_GLOB = COMPARE_DIR / 'compare_C*_seed*.json'

json_files = sorted(glob.glob(str(JSON_GLOB)))
print('Found', len(json_files), 'compare JSON files')

# Target distances to evaluate for time-to-target
TARGET_DISTANCES = [6, 8, 10]


Found 0 compare JSON files


In [21]:
# Section: Perform Calculations with B and C

# Helper: load JSON summaries
def load_compare_json(path):
    with open(path, 'r') as fh:
        return json.load(fh)

runs = [load_compare_json(p) for p in json_files]
print('Loaded', len(runs), 'runs')

# Helper: extract metrics from a run entry (preproc/greedy)
def extract_run_info(entry):
    info = {}
    info['seed'] = entry.get('seed')
    for mode in ['preproc', 'greedy']:
        res = entry.get(mode, {})
        info[f'{mode}_cmd'] = res.get('cmd')
        info[f'{mode}_returncode'] = res.get('result',{}).get('returncode')
        info[f'{mode}_wall_time'] = res.get('result',{}).get('wall_time')
        # try to open associated HDF5 (if recorded by runner env)
        h5f = None
        # If runner created per-seed outputs, look for matching files
        # Fallback: attempt to parse path from stdout for "Saved to <path>"
        stdout = res.get('result',{}).get('stdout','')
        saved = None
        for line in stdout.splitlines()[::-1]:
            if 'Saved to ' in line or 'Saved to' in line:
                saved = line.split('Saved to')[-1].strip()
                break
        info[f'{mode}_hdf5'] = saved
    return info

infos = [extract_run_info(r) for r in runs]
df_info = pd.DataFrame(infos)
display(df_info)


Loaded 0 runs


""


In [22]:
# Section: Validate Results

# Quick checks: how many runs have valid HDF5 paths
valid_preproc = df_info['preproc_hdf5'].notnull() & (df_info['preproc_hdf5']!='')
valid_greedy = df_info['greedy_hdf5'].notnull() & (df_info['greedy_hdf5']!='')
print('Valid preproc hdf5:', valid_preproc.sum(), '/', len(df_info))
print('Valid greedy hdf5:', valid_greedy.sum(), '/', len(df_info))

# Try opening one example HDF5 (first available)
sample = None
for p in df_info['preproc_hdf5'].dropna().unique():
    if p:
        sample = p
        break
if sample:
    print('Sample preproc HDF5:', sample)
    try:
        with h5py.File(sample,'r') as h:
            print('Groups:', list(h.keys()))
    except Exception as e:
        print('Could not open sample HDF5:', e)
else:
    print('No sample preproc HDF5 found in JSONs')


KeyError: 'preproc_hdf5'

In [ ]:
# Section: Visualize B and C

# Helper to load time series of best cost from HDF5 saved by search scripts
# Expected structure in HDF5: group per code, datasets like 'best_cost_times' or 'states' with timestamps

from optimization.experiments_settings import from_edgelist, tanner_graph_to_parity_check_matrix

def _get_primary_group(h5):
    keys = list(h5.keys())
    if not keys:
        return None
    # Choose the single group if available, otherwise the first one
    return h5[keys[0]]


def extract_best_times_from_hdf5(h5path):
    times = []
    costs = []
    attrs = {}
    try:
        with h5py.File(h5path,'r') as h:
            grp = _get_primary_group(h)
            if grp is not None:
                attrs = dict(grp.attrs.items())
            # heuristics: look for datasets named 'best_cost_times' or 'best_cost' or 'states'
            if 'best_cost_times' in h:
                times = np.array(h['best_cost_times'])
                costs = np.array(h['best_cost']) if 'best_cost' in h else np.zeros_like(times)
            else:
                # fallback: if 'states' group exists, use its timestamps or indices
                if 'states' in h:
                    grp_states = h['states']
                    # often there is a dataset 't' or 'time' per state
                    if 'time' in grp_states:
                        times = np.array(grp_states['time'])
                    else:
                        times = np.arange(len(grp_states))
                    # cost per state
                    if 'cost' in grp_states:
                        costs = np.array(grp_states['cost'])
                    else:
                        # try reading attrs for best cost
                        costs = np.zeros_like(times)
                else:
                    # use root attrs as fallback
                    times = np.array([0.0])
                    costs = np.array([h.attrs.get('min_cost', np.nan)])
    except Exception as e:
        print('Error reading', h5path, e)
    return times, costs, attrs


def _safe_float(val):
    try:
        return float(val)
    except Exception:
        return np.nan


def compute_regularity_metrics_from_matrix(H: np.ndarray) -> dict:
    row_w = np.asarray(H.sum(axis=1)).ravel().astype(int)
    col_w = np.asarray(H.sum(axis=0)).ravel().astype(int)

    def _entropy(counts):
        counts = np.asarray(counts, dtype=np.float64)
        total = counts.sum()
        if total <= 0:
            return 0.0
        probs = counts / total
        probs = probs[probs > 0]
        return float(-(probs * np.log2(probs)).sum())

    row_var = float(np.var(row_w)) if row_w.size else 0.0
    col_var = float(np.var(col_w)) if col_w.size else 0.0
    row_entropy = _entropy(np.bincount(row_w)) if row_w.size else 0.0
    col_entropy = _entropy(np.bincount(col_w)) if col_w.size else 0.0

    signatures = set()
    for j in range(H.shape[1]):
        sig = tuple(np.flatnonzero(H[:, j]).tolist())
        signatures.add(sig)

    return {
        "row_weight_var": row_var,
        "col_weight_var": col_var,
        "distinct_col_signatures": int(len(signatures)),
        "row_degree_entropy": row_entropy,
        "col_degree_entropy": col_entropy,
    }


def load_best_state_metrics(h5path):
    try:
        with h5py.File(h5path, 'r') as h:
            grp = _get_primary_group(h)
            if grp is None or 'best_state' not in grp:
                return None
            edges = grp['best_state'][:]
            edge_list = edges[0] if edges.ndim == 2 else edges
            state = from_edgelist(edge_list)
            H = tanner_graph_to_parity_check_matrix(state)
            return compute_regularity_metrics_from_matrix(H)
    except Exception as e:
        print('Error loading best_state metrics from', h5path, e)
        return None


def load_candidate_metrics(h5path, max_rows=2000):
    try:
        with h5py.File(h5path, 'r') as h:
            grp = _get_primary_group(h)
            if grp is None:
                return None
            if 'row_weight_var' not in grp or 'distances_classical' not in grp:
                return None
            row_var = np.array(grp['row_weight_var'])
            col_var = np.array(grp['col_weight_var']) if 'col_weight_var' in grp else np.full_like(row_var, np.nan)
            sig_cnt = np.array(grp['distinct_col_signatures']) if 'distinct_col_signatures' in grp else np.full_like(row_var, np.nan)
            row_ent = np.array(grp['row_degree_entropy']) if 'row_degree_entropy' in grp else np.full_like(row_var, np.nan)
            col_ent = np.array(grp['col_degree_entropy']) if 'col_degree_entropy' in grp else np.full_like(row_var, np.nan)
            d_cl = np.array(grp['distances_classical'])
            d_cl_t = np.array(grp['distances_classical_T']) if 'distances_classical_T' in grp else None
            if d_cl_t is not None:
                min_dist = np.minimum(d_cl, d_cl_t)
            else:
                min_dist = d_cl
            n = len(min_dist)
            if n > max_rows:
                idx = np.linspace(0, n-1, max_rows).astype(int)
                min_dist = min_dist[idx]
                row_var = row_var[idx]
                col_var = col_var[idx]
                sig_cnt = sig_cnt[idx]
                row_ent = row_ent[idx]
                col_ent = col_ent[idx]
            return pd.DataFrame({
                'min_distance': min_dist,
                'row_weight_var': row_var,
                'col_weight_var': col_var,
                'distinct_col_signatures': sig_cnt,
                'row_degree_entropy': row_ent,
                'col_degree_entropy': col_ent,
            })
    except Exception as e:
        print('Error loading candidate metrics from', h5path, e)
        return None


# Build aggregated records
records = []
for idx, row in df_info.iterrows():
    seed = row['seed']
    for mode in ['preproc','greedy']:
        h5p = row.get(f'{mode}_hdf5')
        rec = {'seed': seed, 'mode': mode, 'hdf5': h5p}
        if h5p:
            t, c, attrs = extract_best_times_from_hdf5(h5p)
            rec['times'] = t.tolist()
            rec['costs'] = c.tolist()
            rec['final_cost'] = float(c[-1]) if len(c)>0 else np.nan
            # Prefer time_to_target attr if available
            rec['time_to_target'] = _safe_float(attrs.get('time_to_target', np.nan))
            rec['target_reached'] = bool(attrs.get('target_reached', False))
            # Fallback to total_runtime attr if times are missing
            if len(t) > 0:
                rec['total_time'] = float(t[-1])
            else:
                rec['total_time'] = _safe_float(attrs.get('total_runtime', np.nan))
        else:
            rec['times'] = []
            rec['costs'] = []
            rec['final_cost'] = np.nan
            rec['time_to_target'] = np.nan
            rec['target_reached'] = False
            rec['total_time'] = np.nan
        records.append(rec)

agg_df = pd.DataFrame(records)
display(agg_df[['seed','mode','final_cost','total_time','time_to_target','target_reached']])

# Plot 1: boxplot of final cost by mode
plt.figure(figsize=(8,6))
sns.boxplot(x='mode', y='final_cost', data=agg_df)
plt.title('Final best cost by mode')
plt.ylabel('Final min cost (LER proxy)')
plt.savefig(OUT_DIR / 'box_final_cost_by_mode.png', bbox_inches='tight')
plt.show()

# Plot 2: bar chart median runtime per mode
rt = agg_df.groupby('mode')['total_time'].median().reset_index()
plt.figure(figsize=(8,6))
sns.barplot(x='mode', y='total_time', data=rt)
plt.title('Median total time by mode')
plt.ylabel('Time (s)')
plt.savefig(OUT_DIR / 'median_time_by_mode.png', bbox_inches='tight')
plt.show()

# Plot 3: cumulative-best vs time (overlay seeds)
plt.figure(figsize=(10,6))
for mode in ['preproc','greedy']:
    plt.subplot(1,2,1 if mode=='preproc' else 2)
    subset = agg_df[agg_df['mode']==mode]
    for _, r in subset.iterrows():
        t = np.array(r['times'])
        c = np.array(r['costs'])
        if len(t)==0:
            continue
        # compute cumulative best
        cumbest = np.minimum.accumulate(c)
        plt.plot(t, cumbest, alpha=0.6)
    plt.title(f'Cumulative best — {mode}')
    plt.xlabel('Time (s)')
    plt.ylabel('Best cost so far')
plt.tight_layout()
plt.savefig(OUT_DIR / 'cumulative_best_by_mode.png', bbox_inches='tight')
plt.show()

# Plot 4: ECDF time-to-target using HDF5 attrs
tt_df = agg_df[(~agg_df['time_to_target'].isna()) & (agg_df['time_to_target'] >= 0)]
plt.figure(figsize=(8,6))
for mode, grp in tt_df.groupby('mode'):
    vals = np.sort(grp['time_to_target'].values)
    if len(vals) == 0:
        continue
    y = np.arange(1, len(vals)+1)/len(vals)
    plt.plot(vals, y, marker='o', label=mode)
plt.xlabel('Time to reach target distance (s)')
plt.ylabel('ECDF')
plt.legend()
plt.title('ECDF time-to-target (from HDF5 attrs)')
plt.savefig(OUT_DIR / 'ecdf_time_to_target.png', bbox_inches='tight')
plt.show()

# Regularity metrics for best_state vs LER / time-to-target
metrics_records = []
for _, row in agg_df.iterrows():
    h5p = row.get('hdf5')
    if not h5p:
        continue
    metrics = load_best_state_metrics(h5p)
    if metrics is None:
        continue
    rec = {'seed': row['seed'], 'mode': row['mode'], 'final_cost': row['final_cost'], 'time_to_target': row['time_to_target']}
    rec.update(metrics)
    metrics_records.append(rec)

metrics_df = pd.DataFrame(metrics_records)
display(metrics_df.head())

metric_cols = [
    'row_weight_var',
    'col_weight_var',
    'distinct_col_signatures',
    'row_degree_entropy',
    'col_degree_entropy',
 ]

for metric in metric_cols:
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=metric, y='final_cost', hue='mode', data=metrics_df)
    plt.title(f'Regularity vs LER: {metric}')
    plt.ylabel('Final min cost (LER)')
    plt.savefig(OUT_DIR / f'regularity_{metric}_vs_ler.png', bbox_inches='tight')
    plt.show()

for metric in metric_cols:
    subset = metrics_df[~metrics_df['time_to_target'].isna()]
    if subset.empty:
        continue
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=metric, y='time_to_target', hue='mode', data=subset)
    plt.title(f'Regularity vs time-to-target: {metric}')
    plt.ylabel('Time to target (s)')
    plt.savefig(OUT_DIR / f'regularity_{metric}_vs_time_to_target.png', bbox_inches='tight')
    plt.show()

# Distance vs regularity across candidates (if per-candidate metrics exist)
candidate_frames = []
for _, row in agg_df.iterrows():
    h5p = row.get('hdf5')
    if not h5p:
        continue
    cand_df = load_candidate_metrics(h5p, max_rows=2000)
    if cand_df is None:
        continue
    cand_df['mode'] = row['mode']
    cand_df['seed'] = row['seed']
    candidate_frames.append(cand_df)

if candidate_frames:
    cand_all = pd.concat(candidate_frames, ignore_index=True)
    for metric in metric_cols:
        plt.figure(figsize=(7,5))
        sns.scatterplot(x='min_distance', y=metric, hue='mode', data=cand_all, alpha=0.4)
        plt.title(f'Distance vs regularity: {metric}')
        plt.xlabel('Min distance (classical / transpose)')
        plt.savefig(OUT_DIR / f'distance_vs_{metric}.png', bbox_inches='tight')
        plt.show()



KeyError: "None of [Index(['seed', 'mode', 'final_cost', 'total_time'], dtype='object')] are in the [columns]"